In [ ]:
%pip install uv

In [ ]:
!uv pip install --system \
  trl==0.23.1 \
  accelerate==1.6.0 \
  datasets==3.2.0 \
  bitsandbytes \
  sentencepiece \
  protobuf \
  scipy \
  einops \
  ninja \
  packaging \
  wheel \
  numpy==2.0.2

In [ ]:
!uv pip install --system --no-cache \
  "unsloth==2026.3.7" \
  "unsloth_zoo>=2026.3.4"

In [ ]:
import os
import shutil

from unsloth import FastLanguageModel

from google.colab import drive
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
drive.mount('/content/drive')

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
os.environ['WANDB_MODE'] = 'disabled'

MODEL_ID   = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_ID = "g0ofycatz/OkinDataset"

BASE_DIR   = "/content/drive/MyDrive/okin"
OUTPUT_DIR = os.path.join(BASE_DIR, "okin_llm")
MERGED_DIR  = "/content/merged_model"
GGUF_DIR   = os.path.join(BASE_DIR, "gguf")
QUANT_TYPE = "q4_k_m"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(DATASET_ID, split="train")

def formatting_func(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(formatting_func)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    packing=True,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    num_train_epochs=25,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_total_limit=1,
    save_strategy="no",
    optim="paged_adamw_8bit",
    report_to="none",
)

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=512,
    args=args,
)

trainer.train()

print("Merging LoRA weights...")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

print("Converting to GGUF...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method=QUANT_TYPE,
    temporary_location=MERGED_DIR,
)

print("Cleaning up...")
shutil.rmtree(MERGED_DIR, ignore_errors=True)

print(f"Done. Saved {QUANT_TYPE} GGUF to:", GGUF_DIR)